# Drug Safety Signal Analysis — FDA FAERS

Detecting potential adverse-drug-reaction signals from real-world spontaneous
reports using disproportionality analysis (PRR / ROR).

This notebook walks through the same pipeline as `run_analysis.py`, with
narrative and inline charts. Configure the drug class and quarters in
`src/config.py` first, then run `python -m src.download_faers` to fetch data.

## 1. Setup

In [ ]:
import sys
sys.path.append("..")  # so `import src...` works from the notebooks/ folder

from src.config import TARGET_DRUG_LABEL, TARGET_DRUGS, QUARTERS
from src.load_faers import load_all, build_report_event_table
from src.disproportionality import compute_signals
from src.visualize import plot_top_events, plot_signal_scatter

print("Target:", TARGET_DRUG_LABEL)
print("Drugs:", TARGET_DRUGS)
print("Quarters:", QUARTERS)

## 2. Load and clean the FAERS quarters

`load_all()` reads the `$`-delimited DEMO/DRUG/REAC tables for every quarter in
the config. `build_report_event_table()` de-duplicates cases (latest version
per `caseid`), keeps only suspect drugs (role codes PS/SS), and returns one row
per unique (report, adverse-event) pair with a flag for the target drug.

In [ ]:
tidy = build_report_event_table(load_all())

n_reports = tidy["primaryid"].nunique()
n_target = tidy.loc[tidy["drug_flag"], "primaryid"].nunique()
print(f"Unique reports:                {n_reports:,}")
print(f"Reports with target drug:      {n_target:,}")
print(f"Distinct adverse-event terms:  {tidy['event_pt'].nunique():,}")
tidy.head()

## 3. Compute disproportionality

For each adverse-event term we build the 2x2 table and compute PRR, ROR,
their 95% confidence intervals, and a Yates-corrected chi-squared. A term is
flagged as a signal when **PRR >= 2**, **chi2 >= 4**, and **>= 3 cases** — the
Evans et al. (2001) criteria.

In [ ]:
signals = compute_signals(tidy)
print(f"Flagged {int(signals['is_signal'].sum())} signals "
      f"of {len(signals)} event terms")
signals.head(15)

## 4. Visualise

In [ ]:
%matplotlib inline
plot_top_events(signals);

In [ ]:
plot_signal_scatter(signals);

## 5. Pick one signal to interpret

Choose the most interesting flagged signal — ideally one that is biologically
plausible and not already trivially on the label — and write it up in
`reports/rwe_memo.md`. The memo is what turns this from a stats exercise into
something a pharmacovigilance stakeholder can act on.

In [ ]:
flagged = signals[signals["is_signal"]].sort_values("prr", ascending=False)
flagged.head(10)

## Caveats

Disproportionality flags **statistical** signals, not confirmed causal
relationships. FAERS has no exposure denominator, voluntary/incomplete
reporting, and reporting biases. A signal is a hypothesis to investigate.